# Face Recognition Robust — Deep Embedding (YuNet + SFace)

**Kenapa versi LBPH gagal di lokasi/cahaya berbeda?** LBPH hanya membandingkan histogram tekstur lokal. Begitu cahaya, jarak kamera, atau sudut berubah, jarak ke *semua* orang ikut berubah, dan karena sistemnya "pilih orang terdekat", wajah kamu bisa jatuh ke nama teman.

**Solusi di notebook ini:**
1. **Detektor YuNet + Recognizer SFace** (deep learning, bawaan OpenCV, tanpa dlib/CMake). Wajah diubah jadi *embedding* 128-d yang jauh lebih tahan cahaya, pose, dan kacamata.
2. **Dataset multi-sesi** (lokasi/waktu berbeda) + augmentasi cahaya saat membangun database.
3. **Evaluasi leave-one-condition-out**: model diuji pada kondisi yang *tidak* ada di data latih, lalu **threshold dikalibrasi otomatis**.
4. **Keputusan ketat**: skor harus di atas threshold *dan* unggul jelas dari kandidat kedua (margin), kalau tidak → `Anonymous`.
5. **Smoothing antar-frame** (voting) supaya label tidak berkedip.

Urutan: Cell 1 → 2 → (3 per orang per lokasi) → 4 → 5 → 6.

## Cell 1 — Install & Unduh Model

In [ ]:
# Jalankan sekali saja
%pip install -q opencv-contrib-python numpy

In [ ]:
import os, time, urllib.request
from collections import deque, Counter
import cv2
import numpy as np

MODEL_DIR = "models"
MODEL_URLS = {
    "yunet.onnx": "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    "sface.onnx": "https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
}
os.makedirs(MODEL_DIR, exist_ok=True)
for fname, url in MODEL_URLS.items():
    path = os.path.join(MODEL_DIR, fname)
    if not os.path.exists(path) or os.path.getsize(path) < 100_000:
        print(f"[INFO] Mengunduh {fname} ...")
        urllib.request.urlretrieve(url, path)
    print(f"[OK] {path} ({os.path.getsize(path)/1e6:.1f} MB)")

assert hasattr(cv2, "FaceDetectorYN") and hasattr(cv2, "FaceRecognizerSF"), \
    "OpenCV kamu terlalu lama. Jalankan: pip install -U opencv-contrib-python"
print("OpenCV", cv2.__version__)

## Cell 2 — Konfigurasi & Fungsi Inti
Edit `MEMBERS` sesuai anggota (boleh 4 orang atau lebih). Nama harus sama persis dengan yang dipakai saat koleksi dataset.

In [ ]:
# KONFIGURASI
MEMBERS = ["Devin", "Osa", "Abe"]       # tambahkan nama anggota 

DATASET_DIR  = "dataset_dl"              # dataset baru (foto penuh berwarna)
OUTSIDER_DIR = os.path.join(DATASET_DIR, "_outsiders")   # opsional: foto orang LUAR untuk kalibrasi
DB_PATH      = "face_db.npz"
YUNET_PATH   = os.path.join(MODEL_DIR, "yunet.onnx")
SFACE_PATH   = os.path.join(MODEL_DIR, "sface.onnx")

CAM_INDEX    = 0
DET_SCORE    = 0.85      # minimal keyakinan detektor wajah
MIN_FACE     = 80        # lebar wajah minimal (px) saat koleksi
TOP_K        = 3         # skor orang = rata-rata K kemiripan tertinggi
MARGIN       = 0.03      # selisih minimal skor terbaik vs kandidat kedua
DEFAULT_THRESHOLD = 0.40 # dipakai jika belum dikalibrasi

# MODEL
detector   = cv2.FaceDetectorYN.create(YUNET_PATH, "", (640, 480), DET_SCORE, 0.3, 5000)
recognizer = cv2.FaceRecognizerSF.create(SFACE_PATH, "")


def open_camera():
    cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(CAM_INDEX)
    if not cap.isOpened():
        raise RuntimeError("Webcam tidak bisa dibuka. Coba ganti CAM_INDEX atau tutup aplikasi lain.")
    return cap


def detect_faces(frame):
    """Return list baris [x,y,w,h, 5 landmark(x,y), score]."""
    h, w = frame.shape[:2]
    detector.setInputSize((w, h))
    _, faces = detector.detect(frame)
    return [] if faces is None else list(faces)


def embed_face(frame, face):
    """Align wajah via landmark -> embedding 128-d ter-normalisasi (cosine = dot product)."""
    aligned = recognizer.alignCrop(frame, face)
    feat = recognizer.feature(aligned).flatten().astype(np.float32)
    return feat / (np.linalg.norm(feat) + 1e-9)


def largest_face(faces):
    return max(faces, key=lambda f: f[2] * f[3])


def light_variants(img):
    """Frame asli, mirror, dan versi gelap/terang (simulasi perubahan cahaya)."""
    yield "orig", img
    yield "flip", cv2.flip(img, 1)
    for g in (0.55, 1.8):
        table = ((np.arange(256) / 255.0) ** g * 255).astype("uint8")
        yield f"gamma{g}", cv2.LUT(img, table)


def person_score(q, G, k=TOP_K):
    """Skor kemiripan query(s) q ke galeri G = rata-rata k cosine tertinggi."""
    sims = np.atleast_2d(q) @ G.T
    k = min(k, G.shape[0])
    return np.sort(sims, axis=1)[:, -k:].mean(axis=1)


def identify(q, gallery, threshold, margin=MARGIN):
    """Return (nama_atau_None, skor_terbaik, nama_terdekat)."""
    scores = {n: float(person_score(q, G)[0]) for n, G in gallery.items()}
    ranked = sorted(scores.items(), key=lambda kv: -kv[1])
    best_name, best = ranked[0]
    second = ranked[1][1] if len(ranked) > 1 else -1.0
    accepted = best >= threshold and (best - second) >= margin
    return (best_name if accepted else None), best, best_name


def score_to_percent(s, threshold):
    """Skala tampilan: skor = threshold -> 40%, skor 1.0 -> 100%, skor 0 -> 0%."""
    s = float(s)
    if s >= threshold:
        p = 40 + 60 * (s - threshold) / max(1e-6, 1 - threshold)
    else:
        p = 40 * max(0.0, s) / max(threshold, 1e-6)
    return int(max(0, min(100, round(p))))

print("[OK] Model dimuat. Anggota:", MEMBERS)

## Cell 3 — Kumpulkan Dataset (Multi-Sesi, Multi-Kondisi)
**Ini kunci akurasi.** Jalankan untuk **setiap orang di minimal 2–3 sesi berbeda** (lokasi/jam/lampu berbeda, mis. `kamar`, `kampus`, `luar`). Setiap sesi berisi tahap: terang, remang, backlight, miring, menunduk, menengadah, kacamata, ekspresi.

- Atur kondisi sesuai instruksi di layar, tekan **SPASI** untuk mulai tahap, **Q** untuk berhenti.
- Foto tersimpan sebagai frame penuh berwarna (tanpa preprocessing destruktif); wajah yang tidak jelas/ada >1 wajah otomatis dilewati.
- *Opsional:* taruh 20–50 foto orang luar di `dataset_dl/_outsiders/` agar threshold dikalibrasi terhadap "anonymous" yang nyata.

In [ ]:
STAGES = [  # (tag, instruksi, jumlah foto)
    ("terang",       "Cahaya TERANG, wajah depan",                      25),
    ("remang",       "Cahaya REMANG / lampu redup",                     25),
    ("backlight",    "BACKLIGHT (sumber cahaya di belakang kepala)",    25),
    ("miring_kiri",  "Wajah miring / menoleh sedikit ke KIRI",          20),
    ("miring_kanan", "Wajah miring / menoleh sedikit ke KANAN",         20),
    ("menunduk",     "Kepala sedikit MENUNDUK",                         20),
    ("menengadah",   "Kepala sedikit MENENGADAH",                       20),
    ("kacamata",     "PAKAI/LEPAS KACAMATA (bila punya) bebas sudut",   25),
    ("ekspresi",     "Variasi EKSPRESI (senyum, netral, bicara)",       20),
]
CAPTURE_INTERVAL = 0.25   # detik antar foto agar tidak identik
MIN_SHARPNESS    = 30.0   # variance Laplacian wajah; di bawah ini dianggap blur


def collect_dataset(person, session, stages=STAGES):
    """Simpan foto ke dataset_dl/<person>/<person>_<session>-<tahap>_<n>.jpg"""
    session = session.replace("_", "-")
    save_dir = os.path.join(DATASET_DIR, person)
    os.makedirs(save_dir, exist_ok=True)
    cap = open_camera()
    total, quit_all = 0, False
    print(f"[INFO] {person} | sesi '{session}'. SPASI=mulai tahap, Q=berhenti.")

    for tag, instruction, target in stages:
        if quit_all:
            break
        count, started, last, note = 0, False, 0.0, ""
        while count < target:
            ok, frame = cap.read()
            if not ok:
                quit_all = True
                break
            shown = frame.copy()
            faces = detect_faces(frame)

            if started and time.time() - last >= CAPTURE_INTERVAL:
                if len(faces) != 1:
                    note = "Harus tepat 1 wajah di frame"
                else:
                    x, y, w, h = [int(v) for v in faces[0][:4]]
                    crop = cv2.cvtColor(frame[max(0, y):y + h, max(0, x):x + w], cv2.COLOR_BGR2GRAY)
                    if w < MIN_FACE:
                        note = "Wajah terlalu kecil - dekatkan"
                    elif crop.size == 0 or cv2.Laplacian(crop, cv2.CV_64F).var() < MIN_SHARPNESS:
                        note = "Gambar blur - tahan kepala"
                    else:
                        fname = f"{person}_{session}-{tag}_{count:03d}.jpg"
                        cv2.imwrite(os.path.join(save_dir, fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
                        count += 1; total += 1; last = time.time(); note = ""

            for f in faces:
                x, y, w, h = [int(v) for v in f[:4]]
                cv2.rectangle(shown, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(shown, instruction, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            status = f"[{tag}] {count}/{target}" if started else "Tekan SPASI untuk mulai tahap ini"
            cv2.putText(shown, status, (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            if note:
                cv2.putText(shown, note, (10, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.imshow("Koleksi Dataset (SPASI=mulai, Q=berhenti)", shown)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                quit_all = True
                break
            if key == ord(" "):
                started = True

    cap.release()
    cv2.destroyAllWindows()
    print(f"[DONE] {total} foto baru disimpan di '{save_dir}/'")


collect_dataset("Devin", session="rumah")
# collect_dataset("Devin", session="kampus")
# collect_dataset("Osa",   session="rumah")
# collect_dataset("Osa",   session="kampus")
# collect_dataset("Abe",   session="rumah")
# collect_dataset("Abe",   session="kampus")

## Cell 4 — Bangun Database Embedding
Setiap foto diproses: deteksi → align → embedding, termasuk varian mirror dan gelap/terang. Hasil disimpan ke `face_db.npz`.

In [ ]:
def parse_group(fname, person):
    """'Devin_rumah-terang_003.jpg' -> 'rumah-terang' (sesi+tahap = satu kondisi)."""
    stem = os.path.splitext(fname)[0]
    if stem.startswith(person + "_"):
        stem = stem[len(person) + 1:]
    return stem.rsplit("_", 1)[0]


def embed_image_file(path):
    """Embedding semua varian cahaya dari satu foto. Return list (nama_varian, emb)."""
    img = cv2.imread(path)
    out = []
    if img is None:
        return out
    for vname, v in light_variants(img):
        faces = detect_faces(v)
        if faces:
            out.append((vname, embed_face(v, largest_face(faces))))
    return out


def build_gallery(save=True):
    embs, labels, groups, orig = [], [], [], []
    for name in MEMBERS:
        d = os.path.join(DATASET_DIR, name)
        if not os.path.isdir(d):
            print(f"[WARNING] Folder tidak ada: {d}")
            continue
        files = sorted(f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png")))
        used = 0
        for f in files:
            res = embed_image_file(os.path.join(d, f))
            g = parse_group(f, name)
            for vname, e in res:
                embs.append(e); labels.append(name); groups.append(g); orig.append(vname == "orig")
            used += any(v == "orig" for v, _ in res)
        n_groups = len({g for l, g in zip(labels, groups) if l == name})
        print(f"[INFO] {name}: {used}/{len(files)} foto terpakai, {n_groups} kondisi berbeda")
        if n_groups < 4:
            print(f"       ! {name}: variasi kondisi sedikit. Tambah sesi di lokasi/cahaya lain (Cell 3).")

    if len({*labels}) < 2:
        raise RuntimeError("Butuh dataset minimal 2 orang. Jalankan Cell 3 dulu.")
    db = dict(emb=np.array(embs, dtype=np.float32), labels=np.array(labels),
              groups=np.array(groups), orig=np.array(orig, dtype=bool))
    if save:
        np.savez(DB_PATH, threshold=np.float32(DEFAULT_THRESHOLD), **db)
        print(f"[DONE] {len(embs)} embedding disimpan -> '{DB_PATH}'")
    return db


db = build_gallery()

## Cell 5 — Evaluasi & Kalibrasi Threshold Otomatis
Tiap orang diuji dengan **menyembunyikan satu kondisi** (mis. `kampus-backlight`) dari galeri, lalu mencoba mengenalinya. Ini mensimulasikan "lokasi/cahaya yang belum pernah dilihat" — persis masalah di dunia nyata. Hasilnya dipakai untuk memilih threshold.

In [ ]:
def evaluate(db, outsider_dir=OUTSIDER_DIR, save=True):
    emb, labels, groups, orig = db["emb"], db["labels"], db["groups"], db["orig"]
    names = sorted({str(l) for l in labels})
    genuine, wrong_best, y_true, y_pred = [], [], [], []

    for p in names:
        pm = labels == p
        for g in sorted({str(x) for x in groups[pm]}):
            held = pm & (groups == g)
            q = held & orig
            if not q.any():
                continue
            Q = emb[q]
            cols = []
            for n in names:
                gm = (labels == n) & ~held
                cols.append(person_score(Q, emb[gm]) if gm.any() else np.full(len(Q), -1.0))
            S = np.stack(cols, axis=1)
            own = names.index(p)
            genuine.extend(S[:, own])
            wrong_best.extend(np.delete(S, own, axis=1).max(axis=1))
            y_true.extend([own] * len(Q)); y_pred.extend(S.argmax(axis=1))

    genuine, wrong_best = np.array(genuine), np.array(wrong_best)
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = float((y_true == y_pred).mean())
    print(f"[EVAL] Akurasi identifikasi pada kondisi TAK DIKENAL: {acc:.1%} ({len(y_true)} query)")

    cm = np.zeros((len(names), len(names)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    print("\nConfusion matrix (baris = asli, kolom = prediksi):")
    print("            " + "  ".join(f"{n[:8]:>8}" for n in names))
    for i, n in enumerate(names):
        print(f"{n[:10]:>10}  " + "  ".join(f"{v:>8d}" for v in cm[i]))

    # skor orang luar (opsional) ikut sebagai impostor
    impostor = list(wrong_best)
    if os.path.isdir(outsider_dir):
        gallery = {n: emb[labels == n] for n in names}
        n_out = 0
        for f in os.listdir(outsider_dir):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                r = embed_image_file(os.path.join(outsider_dir, f))
                for vname, e in r:
                    if vname == "orig":
                        impostor.append(max(float(person_score(e, G)[0]) for G in gallery.values()))
                        n_out += 1
        print(f"\n[INFO] {n_out} foto orang luar dipakai untuk kalibrasi")
    impostor = np.array(impostor)

    print(f"\nSkor orang SEBENARNYA : median={np.median(genuine):.3f}  p10={np.percentile(genuine,10):.3f}")
    print(f"Skor SALAH/ORANG LUAR : median={np.median(impostor):.3f}  p90={np.percentile(impostor,90):.3f}  max={impostor.max():.3f}")

    # pilih threshold: maksimalkan balanced accuracy (terima anggota, tolak impostor)
    cands = np.unique(np.concatenate([genuine, impostor]))
    best_t, best_bal = DEFAULT_THRESHOLD, -1
    for t in cands:
        bal = ((genuine >= t).mean() + (impostor < t).mean()) / 2
        if bal > best_bal + 1e-9:
            best_bal, best_t = bal, float(t)
    best_t = float(np.clip(best_t, 0.25, 0.70))
    frr = float((genuine < best_t).mean()); far = float((impostor >= best_t).mean())
    print(f"\n[KALIBRASI] THRESHOLD = {best_t:.3f}  (anggota ditolak {frr:.1%}, impostor lolos {far:.1%})")
    if far > 0.05 or frr > 0.15:
        print("[SARAN] Distribusi masih tumpang tindih: tambah sesi di lokasi/cahaya berbeda, dan/atau foto orang luar ke _outsiders/.")
    if save:
        np.savez(DB_PATH, threshold=np.float32(best_t), emb=emb, labels=labels, groups=groups, orig=orig)
        print(f"[DONE] Threshold tersimpan di '{DB_PATH}'")
    return best_t


THRESHOLD = evaluate(db)

## Cell 6 — Real-Time Recognition
Pipeline: **YuNet detect → align → SFace embedding → skor per orang → threshold + margin → voting antar-frame**.

Tombol: **Q** keluar · **+ / -** ubah threshold · **D** debug (tampilkan kandidat terdekat saat "Anonymous").
Persentase tampilan: skor = threshold → 40%, skor 1.0 → 100%.

In [ ]:
class FaceTracker:
    """Pelacak sederhana (nearest-center) + voting identitas beberapa frame terakhir."""
    def __init__(self, history=8, max_age=6):
        self.tracks, self.next_id = {}, 0
        self.history, self.max_age = history, max_age

    def assign(self, centers, widths):
        unused, ids = set(self.tracks), []
        for c, w in zip(centers, widths):
            c = np.array(c, dtype=float)
            best, best_d = None, 0.8 * w
            for tid in unused:
                d = float(np.linalg.norm(c - self.tracks[tid]["c"]))
                if d < best_d:
                    best, best_d = tid, d
            if best is None:
                best = self.next_id; self.next_id += 1
                self.tracks[best] = {"hist": deque(maxlen=self.history), "c": c, "age": 0}
            else:
                unused.discard(best)
            self.tracks[best]["c"], self.tracks[best]["age"] = c, 0
            ids.append(best)
        for tid in unused:
            self.tracks[tid]["age"] += 1
            if self.tracks[tid]["age"] > self.max_age:
                del self.tracks[tid]
        return ids

    def vote(self, tid, name, score):
        self.tracks[tid]["hist"].append((name, score))

    def decision(self, tid):
        hist = self.tracks[tid]["hist"]
        winner = Counter(n for n, _ in hist).most_common(1)[0][0]
        mean_score = float(np.mean([s for n, s in hist if n == winner]))
        return winner, mean_score


def run_realtime(db_path=DB_PATH):
    data = np.load(db_path)
    emb, labels = data["emb"], data["labels"]
    threshold = float(data["threshold"])
    gallery = {str(n): emb[labels == n] for n in np.unique(labels)}
    palette = [(255, 100, 0), (0, 210, 0), (0, 0, 220), (0, 170, 255), (200, 0, 200), (200, 200, 0)]
    colors = {n: palette[i % len(palette)] for i, n in enumerate(sorted(gallery))}
    tracker, debug = FaceTracker(), False

    cap = open_camera()
    print(f"[INFO] Webcam aktif. Threshold={threshold:.2f}. Q=keluar, +/-=threshold, D=debug")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        faces = detect_faces(frame)
        centers = [(f[0] + f[2] / 2, f[1] + f[3] / 2) for f in faces]
        ids = tracker.assign(centers, [f[2] for f in faces])

        for f, tid in zip(faces, ids):
            x, y, w, h = [int(v) for v in f[:4]]
            name, score, nearest = identify(embed_face(frame, f), gallery, threshold)
            tracker.vote(tid, name, score)
            who, mean_score = tracker.decision(tid)

            if who is not None:
                label = f"{who} ({score_to_percent(mean_score, threshold)}%)"
                color = colors.get(who, (200, 200, 200))
            else:
                label = "Anonymous"
                if debug:
                    label += f" [~{nearest} {score_to_percent(score, threshold)}%]"
                color = (130, 130, 130)

            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
            top = max(y - th - 10, 0)
            cv2.rectangle(frame, (x, top), (x + tw + 8, top + th + 10), color, -1)
            cv2.putText(frame, label, (x + 4, top + th + 3), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

        cv2.putText(frame, f"Threshold: {threshold:.2f}  (+/-)  D=debug", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.imshow("Face Recognition", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key in (ord("+"), ord("=")):
            threshold = min(0.95, threshold + 0.02)
        elif key in (ord("-"), ord("_")):
            threshold = max(0.10, threshold - 0.02)
        elif key == ord("d"):
            debug = not debug

    cap.release()
    cv2.destroyAllWindows()
    print(f"[INFO] Selesai. Threshold terakhir: {threshold:.2f} (set manual ke THRESHOLD bila cocok)")


run_realtime()

## Troubleshooting

| Gejala | Penyebab umum | Solusi |
|---|---|---|
| Kamu dikenali sebagai teman | Dataset hanya 1 lokasi/cahaya | Tambah sesi baru di lokasi bermasalah (Cell 3), lalu ulang Cell 4–5 |
| Anggota sering "Anonymous" | Threshold terlalu tinggi / variasi data kurang | Tekan `-` saat real-time, atau tambah sesi; cek hasil Cell 5 |
| Orang luar dikenali sebagai anggota | Threshold terlalu rendah | Tekan `+`, dan masukkan foto orang luar ke `dataset_dl/_outsiders/` lalu ulang Cell 5 |
| Akurasi Cell 5 rendah | Kondisi di dataset belum mencakup kondisi uji | Foto ulang dengan tahap lengkap di tiap lokasi |
| Label berkedip | Wajah kecil / blur | Dekatkan wajah, perbaiki cahaya depan |

**Catatan:** database di `face_db.npz` hanya berisi embedding (bukan foto). Foto di `dataset_dl/` tetap perlu dijaga privasinya.